# Phase 8 — Notebook 08c: Benchmark MiniLM Reranker trên Top-10 Inputs Cố Định

Notebook này đánh giá thực nghiệm việc áp dụng reranker cục bộ `cross-encoder/ms-marco-MiniLM-L-6-v2` đối với Top 10 kết quả retrieval cố định từ Phase 8 Notebook 08b trên tập dữ liệu Golden Dataset V3 (45 cases) và 572 chunks ẩm thực Huế.

In [ ]:
import sys
from pathlib import Path

# Đảm bảo nạp đúng backend module
REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT / "backend"))

import polars as pl
from evaluation.reranker_benchmark import (
    INPUT_SETTINGS,
    load_reranker_benchmark_inputs,
    run_all_reranker_inputs,
    reconcile_reranker_artifacts,
    RESULTS_PATH,
    CASES_PATH,
)

print("Imported reranker benchmark modules successfully.")

## 1. Giới hạn phạm vi Foods và nguyên tắc không tự động chọn winner

- Thử nghiệm này chỉ so sánh `no-rerank` (Top 5 từ Top 10 của 08b) và `minilm` (MiniLM rerank Top 10 -> Top 5).
- Ba input cố định được sử dụng theo thứ tự:
  1. `dense__e5-small-384` (Production baseline control);
  2. `dense__huydang-dek21-embedding-768` (Mô hình dense tiếng Việt);
  3. `hybrid-bm25-weighted__huydang-dek21-embedding-768` (Chẩn đoán xem MiniLM có khắc phục được lỗi ranking ở category `relationship` hay không).
- Notebook không tự động chọn winner hay kích hoạt production pipeline.

In [ ]:
for s in INPUT_SETTINGS:
    print(f"Setting {s.order}: {s.key} ({s.label})")

## 2. Môi trường và thông tin runtime MiniLM

Kiểm tra môi trường thực thi: CPU FP32, phiên bản thư viện Sentence Transformers, Transformers, Torch và model ID.

In [ ]:
import importlib.metadata
import torch

print("PyTorch version:", torch.__version__)
print("Transformers version:", importlib.metadata.version("transformers"))
print("Sentence-Transformers version:", importlib.metadata.version("sentence-transformers"))

## 3. Xác thực Golden V3, 572 chunks và 135 input cố định từ 08b

Nạp và xác thực fail-closed toàn bộ 135 bản ghi fixed Top-10 từ `phase8_sparse_cases.jsonl` và `phase8_sparse_manifest.json` đối chiếu với Golden V3 và 572 chunks chuẩn.

In [ ]:
inputs = load_reranker_benchmark_inputs()
print(f"Loaded {len(inputs.cases)} Golden V3 cases")
print(f"Loaded {len(inputs.chunks_by_id)} canonical chunks")
print(f"Loaded {len(inputs.fixed_cases)} fixed input cases across 3 settings")
print(f"Smoke cases count: {len(inputs.smoke_case_ids)}")

## 4. Kiểm tra các No-Rerank Baseline Controls

Hiển thị ví dụ Top 5 tài liệu trước khi rerank (no-rerank control) cho câu hỏi đầu tiên.

In [ ]:
sample_case = inputs.fixed_cases[0]
print(f"Sample Case: {sample_case.case.case_id} ({sample_case.case.category})")
print(f"Question: {sample_case.case.question}")
print("Pre-rerank Top 5 chunk IDs:", [d.id for d in sample_case.pre_rerank_documents[:5]])

## 5. Giao thức Single-Load Runtime Lifecycle và Warm-up

Mô hình MiniLM được load đúng 1 lần, đo thời gian khởi động (cold load time) và tài nguyên bộ nhớ (RSS checkpoints) tập trung tại pipeline điều phối ở Section 7 theo đúng giao thức single-load.

In [ ]:
print("Configured model: cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Execution protocol: Single-load runtime (one cold load, one warm-up, smoke gate, full run).")

## 6. Giao thức Technical Smoke Check với 10 case Golden V3 Smoke

Giao thức Technical Smoke chạy 10 câu hỏi chuẩn từ `golden_v3_smoke.jsonl` trên input 1 trước khi bắt đầu benchmark chính để bảo vệ an toàn tài nguyên.

In [ ]:
print(f"Smoke case IDs ({len(inputs.smoke_case_ids)}):", inputs.smoke_case_ids)

## 7. Thực thi Benchmark tuần tự trên 3 inputs (Single-load, Warm-up, Smoke & 45 cases x 3 reps)

Pipeline điều phối chạy tuần tự 3 input settings, mỗi input chạy 3 lần lặp (repetitions) để kiểm tra tính ổn định, sau đó ghi kết quả atomic xuống file và đối soát.

In [ ]:
recon = run_all_reranker_inputs(inputs)
print("Reconciliation complete:", recon.complete)
print(f"Summary rows: {recon.summary_rows} (expected 60)")
print(f"Case records: {recon.case_records} (expected 135)")
if recon.errors:
    print("Errors:", recon.errors)

## 8. Phân tích kết quả tổng thể và theo từng danh mục (Category Deltas & Bootstrap)

Hiển thị bảng tổng hợp kết quả overall và kết quả theo 9 danh mục category.

In [ ]:
summary_df = pl.read_csv(str(RESULTS_PATH))
cases_df = pl.read_ndjson(str(CASES_PATH), infer_schema_length=None)

print("=== OVERALL RESULTS ===")
display(summary_df.filter(pl.col("category") == "overall"))

print("=== CATEGORY BREAKDOWN ===")
display(summary_df.filter(pl.col("category") != "overall"))

## 9. Chi tiết các ca Gained / Lost và danh mục Relationship

Kiểm tra các câu hỏi có thay đổi trạng thái tìm kiếm (gained hoặc lost) và tập trung phân tích danh mục `relationship`.

In [ ]:
print("=== GAINED / LOST CASES ===")
display(cases_df.filter(pl.col("hit_change").is_in(["gained", "lost"])))

print("=== RELATIONSHIP CATEGORY CASES ===")
display(cases_df.filter(pl.col("category") == "relationship"))

## 10. Đánh giá độ trễ, tài nguyên bộ nhớ (RSS) và độ ổn định (Stability)

Hiển thị các chỉ số tài nguyên: thời gian cold load, RSS trước/sau load, peak RSS, p50/p95 latency và tính ổn định qua 3 lần lặp.

In [ ]:
display(
    summary_df.filter(
        (pl.col("category") == "overall") & (pl.col("state_key") == "minilm")
    ).select(
        "input_key", "cold_load_ms", "rss_before_load_mb", "rss_after_load_mb",
        "observed_peak_rss_mb", "rerank_p50_ms", "rerank_p95_ms",
    )
)

## 11. Tổng hợp Evidence Flags và Bàn giao Quyết định cho User

Các flag `eligible`, `clear_gain` và `production_safety` là bằng chứng thực nghiệm mô tả khách quan. Không có kết luận winner tự động; kết quả này được bàn giao để Reviewer báo cáo và Người dùng đưa ra quyết định tiếp theo.

In [ ]:
print("Benchmark completed. All evidence persisted to durable artifacts.")
print("Awaiting User decision.")